## Импортируем необходимые библиотеки

In [ ]:
import re
from datetime import datetime

import pandas as pd 
import numpy as np
import geopandas as gpd

import seaborn as sns
import matplotlib.pyplot as plt

import phik
from phik.report import plot_correlation_matrix

pd.set_option('display.max_columns', None)

Прочитаем файл с данными

In [ ]:
df = pd.read_csv('./notebooks/data/listings_preprocessed.csv', low_memory=False)
df.shape

Привяжем объявления к административным районам Москвы. Загрузим GeoJSON-файл с границами районов Москвы и установим систему координат. Затем на основе широты и долготы из датафрейма мы создали GeoDataFrame с точечной геометрией и с помощью пространственного соединения сопоставили каждой точке название района, внутри которого она находится. А также очистим полученные названия районов от лишних слов ("р-н", "район") и посмотрим долю объявлений с заполненным районом.


In [ ]:
raions = gpd.read_file('./notebooks/data/moscow_raions.geojson')[['name', 'geometry']].set_crs(4326)
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs=4326)
joined = gpd.sjoin(pts, raions, how='left', predicate='within')
geo_district = joined[~joined.index.duplicated(keep='first')].reindex(df.index)['name']
geo_district = geo_district.str.replace(r'^(р-н|район)\s+', '', regex=True).str.replace(r'\s+район$', '', regex=True).str.strip()
df['district'] = df['district'].where(df['district'].notna(), geo_district.values)
df['district'].notna().mean()

In [ ]:
df.loc[(df['ceiling_height'] < 2) | (df['ceiling_height'] > 8), 'ceiling_height'] = np.nan
bad_geometry = df['living_area'].fillna(0) + df['kitchen_area'].fillna(0) > df['total_area']
df.loc[bad_geometry, ['living_area', 'kitchen_area']] = np.nan
df = df[df['total_area'] >= 10]
df.shape

Мы заменили на пропуски аномальные значения высоты потолков (менее 2 м и более 8 м), а также обнулили жилую площадь и площадь кухни в случаях, когда их сумма превышала общую площадь. Дополнительно удалили объекты с общей площадью менее 10 кв.м.

### Заполнение пропусков

In [ ]:
text_cols = [c for c in df.columns if df[c].dtype == 'str']
for col in text_cols:
    mask_1 = df[col].str.strip().str.lower().isin(['', 'nan', 'none'])
    mask_2 = df[col].isna()
    df.loc[mask_1 | mask_2, col] = np.nan 

for c in ['municipality', 'region', 'district', 'parking', 'window_view', 'renovation', 'building_type', 'flat_type', 'seller_user_type', 'room_type']:
    df[c] = df[c].fillna('unknown')

for c in ['deal_conditions', 'seller_type']:
    df[c] = df[c].fillna(df[c].mode()[0])

b_map = {'True': 1, 'False': 0, True: 1, False: 0, 't': 1, 'f': 0}
for c in ['is_apartments', 'is_new_building', 'phone_protected', 'is_studio', 'mortgage_allowed', 'nearest_metro_walk', 'demolished_in_renovation', 'is_penthouse', 'seller_is_owner']:
    df[c] = df[c].map(b_map).fillna(0).astype('int8')

df.shape

Заполнили пропуски в текстовых столбцах, а также проставили unknown для данных, для которых не можем получить достоверную "альтернативу", заполнили модой необходимые поля и привели булевы строки к 0/1.

In [ ]:
df.loc[df['is_studio'] == 1, 'rooms'] = 0
df['rooms'] = df['rooms'].fillna(df['rooms'].median())
df.shape

In [ ]:
group_cols = ['building_type', 'year_built', 'total_floors']

ch_group = df.groupby(group_cols)['ceiling_height'].transform('median')
ch_glob = df['ceiling_height'].median()

df['ceiling_height'] = df['ceiling_height'].fillna(ch_group).fillna(ch_glob)

Мы сгруппировали объявления по типу здания, году постройки и количеству этажей, и для каждой группы вычислили медианное значение высоты потолков. Затем заполнили пропуски в высоте потолков сначала групповой медианой, а затем глобальной медианой по всему датафрейму.

In [ ]:
df['area_bin'] = (df['total_area'] // 5) * 5

for col in ['kitchen_area', 'living_area']:
    med_by = df.groupby(['building_type', 'area_bin'])[col].transform('median')
    df[col] = df[col].fillna(med_by).fillna(df[col].median())

df = df.drop(columns='area_bin')

df.loc[df['year_built'] < 1500, 'year_built'] = np.nan
yb_btype = df.groupby('building_type')['year_built'].transform('median')
yb_dist = df.groupby(df['district'].where(df['district'] != 'unknown'))['year_built'].transform('median')
yb_muni = df.groupby(df['municipality'].where(df['municipality'] != 'unknown'))['year_built'].transform('median')
yb_global = df['year_built'].median()

df['year_built'] = df['year_built'].fillna(yb_btype).fillna(yb_dist).fillna(yb_muni).fillna(yb_global).astype('int32')

Для кухонной и жилой площади мы заполнили пропуски медианой по группам "тип здания + группа по общей площади", а оставшиеся — глобальной медианой. Для года постройки мы очистили значения младше 1500 года и заполнили пропуски последовательно: медианой по типу здания, району, муниципалитету и, наконец, общей медианой.

### Создаём признаки

In [ ]:
df['building_age'] = datetime.now().year - df['year_built']
df['completion_date'] = df['completion_date'].str.extract(r'(\d{4})').astype(float)

current_year = datetime.now().year
df['is_ready'] = ~((df['is_new_building'] == 1) & (df['completion_date'] > current_year))

df['completion_year'] = df['completion_date']
df['years_to_completion'] = df['completion_date'] - current_year
df['is_presale'] = (df['completion_date'] > current_year).astype('int8')
df['has_completion'] = df['completion_date'].notna().astype('int8')

df['had_discount'] = (df['price_last'] < df['price_first']).astype('int8')
df['price_drop_pct'] = (df['price_first'] - df['price_last']) / df['price_first']
df['price_range_pct'] = (df['price_max'] - df['price_min']) / df['price_first']


Мы рассчитали возраст здания, извлекли год завершения строительства и создали бинарные признаки: готово ли здание, продаётся ли на этапе строительства, известна ли дата завершения. Также мы добавили признаки о динамике цены: была ли скидка и процент её падения.

In [ ]:
# признаки этажности
df['is_first_floor'] = (df['floor'] == 1).astype('int8')
df['is_last_floor'] = (df['floor'] == df['total_floors']).astype('int8')
df['floor_ratio'] = (df['floor'] / df['total_floors']).replace([np.inf, -np.inf], np.nan)

# эффективность планировки
df['living_to_total'] = df['living_area'] / df['total_area']
df['kitchen_to_total'] = df['kitchen_area'] / df['total_area']
df['area_per_room'] = df['total_area'] / df['rooms'].replace(0, 1)  # студии считаем за 1 комнату, иначе делим на 0

# относительная цена по локации с учётом комнатности
for geo in ['district', 'municipality']:
    med = df.groupby([geo, 'rooms'])['price_per_m2'].transform('median')
    df[f'ppm2_to_{geo}'] = df['price_per_m2'] / med

df['nearest_metro_time'] = df['nearest_metro_time'].fillna(df['nearest_metro_time'].median())
df['total_lifts'] = df['passenger_lifts'].fillna(0) + df['cargo_lifts'].fillna(0)
df['has_lift'] = (df['total_lifts'] > 0).astype('int8')

Добавили признаки этажности (первый/последний этаж, отношение этажа к общему числу), эффективности планировки (доли жилой и кухонной площади, площадь на комнату) и относительной цены к медиане по району/муниципалитету и заполнили пропуски времени до метро.

In [ ]:
def count_token(value, token):
    if pd.isna(value):
        return -1
    m = re.search(r'(\d+)\s*' + token, value)
    return int(m.group(1)) if m else 0

df['bath_separate'] = df['bathrooms'].map(lambda v: count_token(v, 'разд'))
df['bath_combined'] = df['bathrooms'].map(lambda v: count_token(v, 'совм'))
df['balcony_count'] = df['balcony'].map(lambda v: count_token(v, 'балк'))
df['loggia_count'] = df['balcony'].map(lambda v: count_token(v, 'лодж'))

### Удаляем "лишние" столбцы

In [ ]:
drop_cols = [
    'cian_id',
    'last_seen', 'publication_date',
    'completion_date',
    'developer', 'residential_complex',
    'year_built', 'price',
    'price_min', 'price_max',
    'passenger_lifts', 'cargo_lifts',
    'bathrooms', 'balcony',
    'photos_count', 'views_today',
]
df = df.drop(columns=drop_cols)
df.columns

Сформируем итоговый датасет

In [ ]:
df.to_csv('./notebooks/data/data_final.csv', index=False)
df